In [12]:
import pandas as pd

# Load training data
df = pd.read_csv("../data/train.csv")

# ===== Repeat minimal preprocessing (same as Task 1) =====

import numpy as np

# Replace invalid values
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)

if "thalch" in df.columns:
    df["thalch"] = df["thalch"].replace(0, np.nan)

# Define columns
num_cols = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
cat_cols = ["sex", "dataset", "cp", "fbs", "restecg", "exang", "slope", "thal"]

# Impute
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Drop id
if "id" in df.columns:
    df = df.drop(columns=["id"])

# Encode
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Split
X = df_encoded.drop("target", axis=1)
y = df_encoded["target"]

print("Data ready:", X.shape, y.shape)

Data ready: (736, 21) (736,)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Define only the true continuous/numeric columns to scale
num_cols = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]

# Scale only numeric columns, leave one-hot encoded categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols)
    ],
    remainder="passthrough"
)

# Pipeline links scaling and model so scaling is applied correctly during later training/validation
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

print("Issue 6 complete: scaling pipeline created successfully.")
print("Columns to be scaled:", num_cols)

Issue 6 complete: scaling pipeline created successfully.
Columns to be scaled: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']


In [14]:
# Fit once just to inspect (not for evaluation)
X_temp = logreg_pipeline.named_steps["preprocessor"].fit_transform(X)

print("Transformed shape:", X_temp.shape)

Transformed shape: (736, 21)
